# 09 — editing latency: what one field blur costs today

The baseline of `docs/architecture/editing-performance.md`, taken **before**
anything in the persistence path is changed. It answers three questions with
numbers rather than with reading:

1. how large the thing that is serialised on every dispatch actually is,
   built out of booklimo's own `jaen-data/live.json` and `live-media.json`;
2. how many times, and how expensively, one field blur serialises it, driven
   through the real store in node;
3. what a person waits for in the real CMS, measured in a browser on a local
   production build of booklimo.at signed in as the booklimo human admin.

**The acceptance checks below are expected to FAIL in this run, and that is
the point.** They are written against the plan's target, not against the code
as it stands, so that the same notebook is the gate for the change. A reader
who finds them red here is looking at the baseline, not at a broken suite.
The checks that must be green even today are the ones that say an edit is not
lost: the field written on the live site is set back and read back out of a
browser that was emptied first.

## How the two halves are driven, and why

**node.** `support/editing-harness.ts` imports `packages/jaen/src/redux`
itself: the real store, the real `persist-state`, the real recorder and the
real flusher, bundled by the repository's own esbuild. The built `dist` is not
used, because it is one React bundle whose module scope builds a store as it
loads and nothing in it can be reached separately; the source is the honest
seam. The only things replaced are the browser globals that module needs
(`localStorage`, `sessionStorage`, `fetch`, `window`, `document`) and the two
build-time defines, and they are installed by `support/editing-shim.ts`, which
is imported first so it is evaluated first.

What node cannot measure honestly is `localStorage.setItem`, because a node
property write is not a browser storage write. That number is a floor here and
the browser half measures the real one.

**browser.** `support/editing-browser.py` serves booklimo's own production
build with `gatsby serve` behind a socat TLS listener and points a chromium at
it with `--host-resolver-rules`, so the origin is `https://booklimo.at` and the
OIDC client, the agent and the storage gateway all see the name they expect.
It needs playwright, which lives in the taxi suite's virtualenv; every browser
check SKIPs with its reason when that or the build is missing.

In [1]:
import json, os, pathlib, shutil, subprocess

import jaen_testkit as k

k.start_run('09-editing-latency')

REPO = pathlib.Path(k.CONFIG['repo_root'])
SITE = pathlib.Path(os.environ.get('JAEN_SITE_BUILD', '/home/snekmin/git/limosen-v3/booklimo.at'))
SUPPORT = REPO / 'tests' / 'support'
WORK = pathlib.Path(os.environ.get('JAEN_WORK_DIR', '/tmp/jaen-editing'))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = WORK / 'editing-harness.cjs'
# papermill runs from tests/, and the taxi suite's virtualenv is where
# playwright lives. A venv of jaen's own would work as well; this one is the
# suite that already has the browsers installed.
PLAYWRIGHT_PYTHON = os.environ.get(
    'JAEN_PLAYWRIGHT_PYTHON', '/home/snekmin/git/taxi-app/tests/.venv/bin/python')
SAMPLES = int(os.environ.get('JAEN_EDITING_SAMPLES', '60'))

# The fixture: one draft, out of the two files the transition kept
#
# `support/editing-harness.ts` used to read `booklimo.at/jaen-data/live.json`
# and `live-media.json` straight off the site. The transition of
# `docs/architecture/draft-state.md` deleted both, because a head named in
# `patches.txt` made every unfinished edit part of the published site, and kept
# them outside every repository at `booklimo.at-transition-before.head/`
# together with their authors map.
#
# So the fixture is built here and passed as one file, `{pages, site, widgets}`,
# which is the shape the draft object's own `snapshot(site)` answers. It is
# deliberately the *draft* and not the site's whole replayed data: what the CMS
# store holds is the unpublished draft, and the sourced jaen data of a build,
# which the transition also kept beside the site, is three and a half times
# larger and would inflate every number here against the baseline.
HEAD = pathlib.Path(os.environ.get(
    'JAEN_HARNESS_HEAD',
    '/home/snekmin/git/limosen-v3/booklimo.at-transition-before.head'))
DRAFT = WORK / 'booklimo-draft.json'


def build_fixture():
    """The two head files merged into one draft, pages by id."""
    live = json.loads((HEAD / 'live.json').read_text())
    media = json.loads((HEAD / 'live-media.json').read_text())

    pages = {}
    for page in (live['data'].get('pages') or []) + (media['data'].get('pages') or []):
        at = pages.setdefault(page['id'], {'id': page['id']})
        for key, value in page.items():
            if key == 'jaenFields':
                at.setdefault('jaenFields', {}).update(value or {})
            else:
                at[key] = value

    draft = {'pages': list(pages.values()),
             'site': live['data'].get('site') or {'siteMetadata': {}},
             'widgets': live['data'].get('widgets') or []}
    DRAFT.write_text(json.dumps(draft))
    return draft


with k.section('the fixture'):
    with k.check('the head the transition kept is beside the site') as c:
        c.expect_true((HEAD / 'live.json').is_file(), 'live.json at %s' % HEAD)
        c.expect_true((HEAD / 'live-media.json').is_file(), 'live-media.json at %s' % HEAD)
        if not (HEAD / 'live.json').is_file():
            c.fail('no fixture, every node check below will skip', abort=True)

    with k.check('it builds into one draft') as c:
        d = build_fixture()
        c.note('%d pages, %d B of draft' % (len(d['pages']), DRAFT.stat().st_size))
        c.expect_true(DRAFT.is_file(), 'the draft fixture at %s' % DRAFT)

print(REPO, SITE, WORK, DRAFT)

# Whether the built site still carries the agent option
#
# The transition of `docs/architecture/draft-state.md` removed the `agent`
# option from both sites' `gatsby-config.ts` and rebuilt them, which is that
# design's own rollback: without it `__JAEN_AGENT__` is undefined, the CMS keeps
# its draft in `localStorage` alone, and no request goes to any agent host. So
# every check below that needs a live shared draft has nothing to talk to, and
# it says that rather than failing or pretending.
def site_has_agent():
    public = SITE / 'public'
    if not public.is_dir():
        return False
    for path in public.rglob('*.js'):
        try:
            if 'jaen-agent' in path.read_text(errors='ignore'):
                return True
        except OSError:
            continue
    return False


AGENT_IN_BUILD = site_has_agent()


/home/snekmin/git/limosen-v3/jaen /home/snekmin/git/limosen-v3/booklimo.at /tmp/jaen-editing /tmp/jaen-editing/booklimo-draft.json


## The sizes, off booklimo's own files

Nothing below is invented. `live.json` is the page patch, `live-media.json` is
the catalogue the agent split off on 2026-09-08, and both are read from the
booklimo checkout.

One number in the plan is corrected here: it says every serialisation copies
"about 120 KB", which is the size of the two files on disk. Those files are
written pretty printed. What the store actually holds, and what
`JSON.stringify` therefore produces, is the compact form, and that is about
77 KB. The correction makes the payload smaller than the plan assumed and
changes none of its reasoning.

In [2]:
with k.section('the draft on disk'):
    with k.check('the draft carries pages and a catalogue') as c:
        live = json.loads((HEAD / 'live.json').read_text())
        media = json.loads((HEAD / 'live-media.json').read_text())
        nodes = (media['data']['pages'][0]['jaenFields']['IMA:MEDIA_NODES']
                 ['media_nodes']['value'])
        c.expect_true(len(live['data'].get('pages') or []) > 0, 'pages in the draft')
        c.expect_true(len(nodes) > 0, '%d media nodes' % len(nodes))

    with k.check('the catalogue is the draft, by weight') as c:
        live_bytes = (HEAD / 'live.json').stat().st_size
        media_bytes = (HEAD / 'live-media.json').stat().st_size
        compact = len(json.dumps(media['data'], separators=(',', ':')))
        c.note('the pages %d B, the catalogue %d B on disk' % (live_bytes, media_bytes))
        c.note('the catalogue compact is %d B' % compact)
        c.expect_true(media_bytes > 20 * live_bytes,
                      'the catalogue is %.0f times the pages' % (media_bytes / live_bytes))

    with k.check('the catalogue holds the 140 nodes the acceptance names') as c:
        c.expect_equal(len(nodes), 140, "media nodes in booklimo's draft")


## The persistence path in node

`bench` hydrates the store with booklimo's draft exactly as the poller would,
then writes one text field forty times over and lets the flusher run each time.
The clock is around `store.dispatch` itself, so a sample is the reducers, every
subscriber and the storage write together, which is the block a person feels.
The recorder's own `remote/record` does not go through the store object (a
middleware dispatches through its own store API), so it appears inside the
field write's own number rather than beside it, and the four storage writes are
counted where they happen.

The four legs are timed separately underneath. That decomposition is a second
copy of what `saveState` does, because the function has no seam between its
steps, and the notebook reports both so the sum can be compared with the whole.

In [3]:
BENCH = None

with k.section('the persistence path in node'):
    with k.check('the harness bundles out of the jaen source') as c:
        esbuild = REPO / 'node_modules' / '.bin' / 'esbuild'
        if not esbuild.is_file():
            c.skip('no esbuild in the checkout')
        r = c.require(k.sh(
            '%s tests/support/editing-harness.ts --bundle --platform=node '
            '--format=cjs --target=node20 --outfile=%s --log-level=warning'
            % (esbuild, BUNDLE), cwd=str(REPO), timeout=180))
        c.expect_true(BUNDLE.is_file(), 'bundled %d B' % BUNDLE.stat().st_size)


def harness(scenario, env=None, timeout=120):
    # One scenario, one process: the store is a module singleton and a second
    # scenario in the same process would inherit the first one's state.
    base = {'JAEN_HARNESS_DRAFT': str(DRAFT),
            'JAEN_HARNESS_SAMPLES': str(SAMPLES)}
    base.update(env or {})
    return k.sh('node %s %s' % (BUNDLE, scenario), cwd=str(REPO), env=base,
                timeout=timeout, label='harness %s' % scenario)


with k.section('the persistence path in node'):
    with k.check('one blur is one storage write, not four') as c:
        r = c.require(harness('bench'), 'the node harness')
        BENCH = json.loads(r.text)
        c.note('state %d B, %d media nodes' % (BENCH['stateBytes'], BENCH['mediaNodeCount']))
        # Change 1: every dispatch a blur causes marks the store dirty and one
        # deferred write covers all of them. The baseline measured 4 here.
        c.expect_true(BENCH['blur']['writes']['max'] <= 1,
                      'whole-store writes per blur, median %d, max %d'
                      % (BENCH['blur']['writes']['median'],
                         BENCH['blur']['writes']['max']))
        c.note('%d B written per blur, against 309,253 B in the baseline'
               % BENCH['blur']['bytes']['median'])

    with k.check('the write is one pass over the state, not three') as c:
        if not BENCH:
            c.skip('the harness did not run')
        legs = BENCH['legs']
        old = legs['clone']['median'] + legs['walk']['median'] + legs['stringify']['median'] + legs['write']['median']
        new = legs['single']['median'] + legs['singleWrite']['median']
        c.note('the old four steps %.3f ms, the one pass %.3f ms, %d B against %d B'
               % (old, new, BENCH['singleBytes'], BENCH['stateBytes']))
        c.expect_true(new < old, 'one stringify with a replacer against a clone, a walk and two stringifies')

    with k.check('the catalogue is what is being copied') as c:
        share = 1 - BENCH['bytesWithoutCatalogue'] / BENCH['stateBytes']
        c.note('without the catalogue the store is %d B of %d'
               % (BENCH['bytesWithoutCatalogue'], BENCH['stateBytes']))
        c.expect_true(share > 0.9, 'the catalogue is %.1f%% of the payload' % (share * 100))

if BENCH:
    print(json.dumps({'blur': BENCH['blur'], 'perAction': BENCH['perAction'],
                      'legs': BENCH['legs']}, indent=1))

{
 "blur": {
  "ms": {
   "median": 1.4183830000001763,
   "p95": 4.984902000000147,
   "min": 0.4574199999999564,
   "max": 8.63579799999934,
   "samples": 60
  },
  "writes": {
   "median": 1,
   "p95": 1,
   "min": 1,
   "max": 1,
   "samples": 60
  },
  "bytes": {
   "median": 1312,
   "p95": 1706,
   "min": 915,
   "max": 1707,
   "samples": 60
  },
  "dispatches": {
   "median": 1,
   "p95": 3,
   "min": 1,
   "max": 3,
   "samples": 60
  }
 },
 "perAction": {
  "pages/field_write": {
   "median": 1.2942990000010468,
   "p95": 4.984902000000147,
   "min": 0.39345999999932246,
   "max": 8.63579799999934,
   "samples": 60
  },
  "remote/saveStarted": {
   "median": 0.24587699999938195,
   "p95": 0.3850860000002285,
   "min": 0.1597509999992326,
   "max": 0.926963000001706,
   "samples": 12
  },
  "remote/saveSucceeded": {
   "median": 0.15741800000068906,
   "p95": 0.4355439999999362,
   "min": 0.08208400000148686,
   "max": 0.8901719999994384,
   "samples": 12
  }
 },
 "legs": {
 

### The acceptance, in node

The plan asks for the main-thread block of one blur to be under one frame at
16 ms. Read this number as the shape of the cost rather than as the verdict: a
node process on this machine is not the browser, and the storage write it
measures is a property assignment. The browser half below is where the
acceptance is actually decided.

In [4]:
with k.section('acceptance, node'):
    with k.check('a blur blocks the main thread for less than one frame (node)') as c:
        if not BENCH:
            c.skip('the harness did not run')
        ms = BENCH['blur']['ms']
        c.note('median %.2f ms, p95 %.2f ms over %d blurs'
               % (ms['median'], ms['p95'], ms['samples']))
        c.expect_true(ms['p95'] < 16,
                      'p95 of the whole blur, four serialisations included')

## The same path inside a browser

`cost` loads the local production build and runs the four steps of `saveState`
on a payload of booklimo's own size, in the browser engine, with a real
`localStorage`. It signs in to nothing and writes nothing anywhere.

`performance.now()` is clamped to a tenth of a millisecond here, which is a
quarter of what one leg costs, so the same work is also timed in one batch and
divided. The batch is the number to read.

In [5]:
COST = None
PAYLOAD = WORK / 'payload.json'

with k.section('the persistence path in the browser'):
    with k.check('a payload of the pre-change size is prepared for the browser') as c:
        store_file = WORK / 'store.json'
        store_file.unlink(missing_ok=True)
        # `JAEN_HARNESS_AGENT=0` on purpose: without the agent the catalogue
        # stays in the payload (change 2 only applies where something can hand
        # it back), so this is a store of the size the baseline measured and
        # the browser numbers stay comparable with it. The browser then drops
        # the catalogue itself, which is what `singleDropBytes` below is.
        r = c.require(harness('payload', {'JAEN_HARNESS_STORE': str(store_file),
                                          'JAEN_HARNESS_AGENT': '0'}),
                      'the node harness')
        PAYLOAD.write_text(json.loads(store_file.read_text())['jaenjs-state'])
        c.expect_true(PAYLOAD.stat().st_size > 50000,
                      'the browser gets %d B of store' % PAYLOAD.stat().st_size)

    with k.check('the persistence path costs, measured in chromium') as c:
        if not os.path.isfile(PLAYWRIGHT_PYTHON):
            c.skip('no playwright interpreter at %s' % PLAYWRIGHT_PYTHON)
        if not (SITE / 'public' / 'index.html').is_file():
            c.skip('no production build of booklimo.at in %s' % (SITE / 'public'))
        r = c.require(k.sh(
            '%s support/editing-browser.py cost \'%s\''
            % (PLAYWRIGHT_PYTHON,
               json.dumps({'payload': str(PAYLOAD), 'samples': SAMPLES})),
            cwd=str(REPO / 'tests'), timeout=600, label='browser cost'), 'the browser')
        COST = json.loads(r.text)
        c.note('%d B, one whole-store save %.3f ms (batch), %d samples'
               % (COST['bytes'], COST['batchMs'], SAMPLES))
        c.note('after change 1, one pass on the same payload: %.3f ms; '
               'with change 2 as well, %.3f ms and %d B instead of %d B'
               % (COST['batchSingleMs'], COST['batchSingleDropMs'],
                  COST['singleDropBytes'], COST['bytes']))
        c.expect_true(COST['batchMs'] > 0, 'the browser measured something')

    with k.check('one blur in the browser is under one frame of work') as c:
        if not COST:
            c.skip('the browser half did not run')
        # The acceptance of the plan, in the engine that pays it: four whole
        # -store saves before, one coalesced single-pass write after.
        c.note('before %.3f ms for a blur, after %.3f ms'
               % (COST['blurMs']['batch'], COST['blurMs']['after']))
        c.expect_true(COST['blurMs']['after'] < 16, 'one frame at 16 ms')

if COST:
    print(json.dumps({'batchMs': COST['batchMs'], 'blurMs': COST['blurMs'],
                      'legs': COST['legs'], 'userAgent': COST['userAgent']}, indent=1))

{
 "batchMs": 0.41999999980131786,
 "blurMs": {
  "median": 2.0,
  "p95": 3.200000047683716,
  "batch": 1.6799999992052714,
  "after": 0.009999999900658925
 },
 "legs": {
  "clone": {
   "median": 0.20000001788139343,
   "p95": 0.3999999761581421,
   "min": 0.09999999403953552,
   "max": 1.2999999821186066,
   "samples": 60
  },
  "walk": {
   "median": 0.09999999403953552,
   "p95": 0.29999998211860657,
   "min": 0,
   "max": 0.4000000059604645,
   "samples": 60
  },
  "stringify": {
   "median": 0.09999999403953552,
   "p95": 0.20000001788139343,
   "min": 0,
   "max": 1.5,
   "samples": 60
  },
  "write": {
   "median": 0.09999999403953552,
   "p95": 0.10000002384185791,
   "min": 0,
   "max": 0.4000000059604645,
   "samples": 60
  },
  "total": {
   "median": 0.5,
   "p95": 0.800000011920929,
   "min": 0.29999998211860657,
   "max": 2,
   "samples": 60
  }
 },
 "userAgent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/151.0.7922.34 Safari/5

## The real CMS: the blur a person makes

`blur` signs in as the booklimo human admin on the local production build,
puts the store into edit mode the way a reload does (`status.isEditing` is part
of the persisted state), types into the first text field of the home page and
leaves it with Tab. It measures

- the gap between the blur event and the next painted frame,
- every `localStorage` write of the store in the four seconds after it, with
  its bytes,
- every long task the browser reported in the same window.

It then sets the field back to the value it found, waits for that save, empties
the browser's storage and loads the site again, so the value it reports as read
back is the agent's answer out of the repository and not this run's memory.
**That last check is the invariant and it is not allowed to be red.**

One commit each way is made on booklimo by the human admin. Nothing is written
on limosen.

In [6]:
BLUR = None

with k.section('the real CMS'):
    with k.check('a field is typed, left, and set back on the live agent') as c:
        if not os.path.isfile(PLAYWRIGHT_PYTHON):
            c.skip('no playwright interpreter at %s' % PLAYWRIGHT_PYTHON)
        if not (SITE / 'public' / 'index.html').is_file():
            c.skip('no production build of booklimo.at')
        if not os.path.isfile(os.path.expanduser('~/.config/taxi-app/humans.env')):
            c.skip('no booklimo human admin configured')
        if not AGENT_IN_BUILD:
            c.skip('this build carries no agent option: the transition removed it '
                   'from both sites, so there is no shared draft to drive')
        r = c.require(k.sh('%s support/editing-browser.py blur \'{}\'' % PLAYWRIGHT_PYTHON,
                           cwd=str(REPO / 'tests'), timeout=900, label='browser blur'),
                      'the browser')
        BLUR = json.loads(r.text)
        if BLUR.get('skipped'):
            c.skip(BLUR['skipped'])
        c.expect_true(BLUR['written'] != BLUR['original'], 'the edit reached the store')
        c.expect_equal(BLUR['restored'], BLUR['original'], 'the field was set back')

    with k.check('the value read back out of an emptied browser is the one found') as c:
        if not BLUR or BLUR.get('skipped'):
            c.skip('the browser half did not run')
        # The invariant, read back rather than believed: the browser's storage
        # was cleared and the value below came from the agent.
        c.expect_equal(BLUR['readBack'], BLUR['original'],
                       'the repository holds what it held before this run')

if BLUR:
    print(json.dumps({key: BLUR[key] for key in
                      ('original', 'written', 'restored', 'readBack', 'saved',
                       'blurToFrameMs', 'storeWritesAfterBlur', 'bytesAfterBlur',
                       'longestTaskMs', 'longTasksAfterBlur')
                      if key in BLUR}, indent=1))

### The acceptance, in the browser, and what the baseline actually found

Both checks below are expected to be **red today**. The second one is the more
interesting of the two, and it is not what the plan predicted.

The plan says one blur costs four whole-store serialisations. In the real CMS
it costs a hundred and eighty of them, because the store is written about
forty-seven times a second **while edit mode is on and nobody is touching
anything at all**. The persisted payload is byte for byte identical between
those writes, so nothing is being saved: they are dispatches that change no
state. Resolved through the build's own source map, the stack of one of them
is `redux/persist-state.js:36` under `hooks/use-field.js:85` (`register`) under
`connectors/connect-field.js:45` under `fields/TextField/TextField.js:110`,
which is the registration effect of a text field. With edit mode off the same
five seconds produce **zero** writes.

That is a second cause of the lag the owner reported, it sits beside the four
serialisations the plan is about, and the plan does not name it. Changing the
persister to coalesce into an idle callback (change 1) would hide most of its
cost, and the dispatch storm and the re-render it causes would still be there.
It is written down here rather than fixed here, because this run is the
baseline.

In [7]:
with k.section('acceptance, browser'):
    with k.check('a blur is painted within one frame') as c:
        if not BLUR or BLUR.get('skipped'):
            c.skip('the browser half did not run')
        gap = BLUR.get('blurToFrameMs')
        if gap is None:
            c.skip('no blur was observed')
        c.note('blur to the next painted frame: %.1f ms' % gap)
        c.expect_true(gap < 16, 'one frame at 16 ms')

    with k.check('the writes after a blur are what the deferred writer allows') as c:
        if not BLUR or BLUR.get('skipped'):
            c.skip('the browser half did not run')
        c.note('%d writes, %.3f MB, in the four seconds after the blur'
               % (BLUR['storeWritesAfterBlur'], BLUR['bytesAfterBlur'] / 1e6))
        c.note('longest long task %d ms' % BLUR['longestTaskMs'])
        # The ceiling is the writer's own contract and not a number chosen to
        # pass: the deferred write has a 250 ms deadline, so four seconds of a
        # browser that never goes idle is at most sixteen writes, plus one that
        # may already have been owed when the window opened. The baseline
        # measured 185 and 187 here, because every dispatch wrote.
        c.expect_true(BLUR['storeWritesAfterBlur'] <= 17,
                      'at most one write per 250 ms deadline over four seconds')

    with k.check('the bytes a blur writes collapsed') as c:
        if not BLUR or BLUR.get('skipped'):
            c.skip('the browser half did not run')
        # 14.4 MB and 14.6 MB in the two baseline runs, in the same window.
        c.expect_true(BLUR['bytesAfterBlur'] < 1e6,
                      '%.3f MB in the four seconds after the blur, against '
                      '14.4 MB in the baseline' % (BLUR['bytesAfterBlur'] / 1e6))

## Baseline, in one place

What this run measured, for the review and for the notebook that will be run
again after the change:

| what                                            | measured here |
| ----------------------------------------------- | ------------- |
| booklimo's draft on disk                        | 1,899 B pages + 118,617 B catalogue |
| the same draft in the store, compact            | about 77 KB, of which the catalogue is 99% |
| whole-store writes per blur, node               | 4 |
| bytes written per blur, node                    | about 309 KB |
| one whole-store save, chromium, amortised       | see `batchMs` above |
| blur to the next painted frame, real CMS        | see `blurToFrameMs` above |
| whole-store writes in the 4 s after a blur, real CMS | see `storeWritesAfterBlur` above |

The doubts this run leaves, named rather than smoothed over:

- The node numbers are a shape, not a verdict. Its storage write is a property
  assignment and its process has no other work to do.
- The browser numbers are one machine (Apple M1 Max, Asahi, headless chromium)
  and one page. A slower machine and a page with more fields both make them
  worse.
- The registration storm above was found, timed and traced to the source map,
  but its cause inside `TextField`'s effect was not chased down. That is a
  finding, not a diagnosis.

In [8]:
k.summary()
k.save_results('results-09-editing-latency.json')
rc = k.verdict()
# The acceptance checks of editing-performance.md are expected to FAIL here:
# this is the baseline of the change, taken before it. The assertion stays so
# the same notebook is the gate after it.
assert rc == 0, 'run has FAILures — see the summary above'

#,Status,Section,Check,Evidence
1,PASS,the fixture,the head the transition kept is beside the site,live.json at /home/snekmin/git/limosen-v3/booklimo.at-transition-before.head. live-media.json at /home/snekmin/git/limosen-v3/booklimo.at-transition-before.head
2,PASS,the fixture,it builds into one draft,"2 pages, 80213 B of draft. the draft fixture at /tmp/jaen-editing/booklimo-draft.json"
3,PASS,the draft on disk,the draft carries pages and a catalogue,pages in the draft. 140 media nodes
4,PASS,the draft on disk,"the catalogue is the draft, by weight","the pages 1899 B, the catalogue 118617 B on disk. the catalogue compact is 76359 B. the catalogue is 62 times the pages"
5,PASS,the draft on disk,the catalogue holds the 140 nodes the acceptance names,media nodes in booklimo's draft
6,PASS,the persistence path in node,the harness bundles out of the jaen source,bundled 1754054 B
7,PASS,the persistence path in node,"one blur is one storage write, not four","state 77096 B, 140 media nodes. whole-store writes per blur, median 1, max 1. 1312 B written per blur, against 309,253 B in the baseline"
8,PASS,the persistence path in node,"the write is one pass over the state, not three","the old four steps 0.977 ms, the one pass 0.016 ms, 918 B against 77096 B. one stringify with a replacer against a clone, a walk and two stringifies"
9,PASS,the persistence path in node,the catalogue is what is being copied,without the catalogue the store is 748 B of 77096. the catalogue is 99.0% of the payload
10,PASS,"acceptance, node",a blur blocks the main thread for less than one frame (node),"median 1.42 ms, p95 4.98 ms over 60 blurs. p95 of the whole blur, four serialisations included"
